In [2]:
import pandas as pd

# Carregando a tabela orders
RAW_PATH = '../data/raw/'
orders = pd.read_csv(RAW_PATH + 'olist_orders_dataset.csv')

In [3]:
# Colunas de data na tabela orders — precisam ser datetime para análises temporais

colunas_data_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in colunas_data_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
    # errors='coerce' transforma valores inválidos em NaT em vez de dar erro

# Verificação — sempre confirme depois de converter
print("Tipos após conversão:")
print(orders[colunas_data_orders].dtypes)
print(f"\nNovos NaT gerados (valores que falharam na conversão):")
print(orders[colunas_data_orders].isnull().sum())

Tipos após conversão:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Novos NaT gerados (valores que falharam na conversão):
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [4]:
# PASSO 1: Entender o PADRÃO dos nulos antes de decidir o que fazer

# Filtrar a tabela para ver apenas os pedidos SEM data de entrega
nulos_entrega = orders[orders['order_delivered_customer_date'].isnull()]

print("Status dos pedidos sem data de entrega:")
print(nulos_entrega['order_status'].value_counts())

# PASSO 2: Tratamento
# Como o nulo é informativo (o pedido realmente não foi entregue), mantemos os NaT.
# Uma coluna booleana (Verdadeiro/Falso) para facilitar análises futuras:
orders['foi_entregue'] = orders['order_delivered_customer_date'].notna()

print(f"\nColuna 'foi_entregue' criada com sucesso!")

Status dos pedidos sem data de entrega:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Coluna 'foi_entregue' criada com sucesso!


In [5]:
# 1. Dias úteis/corridos prometidos de entrega (Prazo total dado ao cliente)
# Diferença entre a data estimada e a data de compra
orders['dias_prazo_prometido'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.days

# 2. Dias reais de entrega (Quanto tempo demorou de verdade)
# Diferença entre a data real de entrega e a data de compra
orders['dias_entrega_real'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

# 3. Dias de atraso 
# Se o resultado for positivo, atrasou. Se for negativo ou zero, chegou adiantado ou no prazo.
orders['dias_atraso'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# Resumo estatístico rápido dessas novas colunas
print("Resumo das novas métricas de tempo (em dias):")
print(orders[['dias_prazo_prometido', 'dias_entrega_real', 'dias_atraso']].describe().round(1))

Resumo das novas métricas de tempo (em dias):
       dias_prazo_prometido  dias_entrega_real  dias_atraso
count               99441.0            96476.0      96476.0
mean                   23.4               12.1        -11.9
std                     8.8                9.6         10.2
min                     1.0                0.0       -147.0
25%                    18.0                6.0        -17.0
50%                    23.0               10.0        -12.0
75%                    28.0               15.0         -7.0
max                   155.0              209.0        188.0


In [6]:
# ── COLUNA 3: flag de pedido atrasado ────────────────────────────────────────
# Cria uma coluna que diz apenas True (Atrasou) ou False (No prazo/Adiantado)
orders['pedido_atrasado'] = orders['dias_atraso'] > 0

# ── COLUNA 4: extrair componentes temporais para análise de sazonalidade ─────
orders['ano_compra'] = orders['order_purchase_timestamp'].dt.year
orders['mes_compra'] = orders['order_purchase_timestamp'].dt.month
orders['dia_semana_compra'] = orders['order_purchase_timestamp'].dt.day_name()
orders['hora_compra'] = orders['order_purchase_timestamp'].dt.hour
orders['trimestre_compra'] = orders['order_purchase_timestamp'].dt.quarter

# ── VERIFICAÇÃO ───────────────────────────────────────────────────────────────
print(f"Pedidos atrasados: {orders['pedido_atrasado'].sum():,} "
      f"({orders['pedido_atrasado'].mean()*100:.1f}%)")

Pedidos atrasados: 6,535 (6.6%)


In [7]:
products = pd.read_csv(RAW_PATH + 'olist_products_dataset.csv')
product_category_translation = pd.read_csv(RAW_PATH + 'product_category_name_translation.csv')

print("Tabelas de produtos carregadas para limpeza.")

# ── 1. Traduzir categorias para inglês ──────────────────────────────────────
products = products.merge(
    product_category_translation,
    on='product_category_name',
    how='left'
)

# Se alguma categoria ficou sem tradução, mantemos o nome original em português
products['product_category_name_english'] = (
    products['product_category_name_english']
    .fillna(products['product_category_name'])
)

# ── 2. Verificar nulos nas dimensões físicas (antes do tratamento) ─────────
cols_dimensao = ['product_weight_g', 'product_length_cm', 
                 'product_height_cm', 'product_width_cm']

print("\nQuantidade de nulos antes do tratamento por mediana:")
print(products[cols_dimensao].isnull().sum())

# ── 3. Tratamento Avançado: Preenchimento com a mediana da categoria ───────
# A mediana é robusta contra outliers e usar a da categoria evita misturar o peso de uma 'geladeira' com o de um 'brinco'
for col in cols_dimensao:
    products[col] = products.groupby('product_category_name_english')[col].transform(
        lambda x: x.fillna(x.median())
    )

print("\nQuantidade de nulos após o tratamento:")
print(products[cols_dimensao].isnull().sum())

Tabelas de produtos carregadas para limpeza.

Quantidade de nulos antes do tratamento por mediana:
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

Quantidade de nulos após o tratamento:
product_weight_g     610
product_length_cm    610
product_height_cm    610
product_width_cm     610
dtype: int64


In [8]:
products = pd.read_csv(RAW_PATH + 'olist_products_dataset.csv')
products = products.merge(product_category_translation, on='product_category_name', how='left')

# 2. A CORREÇÃO: Preencher categorias nulas ANTES do agrupamento
products['product_category_name_english'] = (
    products['product_category_name_english']
    .fillna(products['product_category_name'])
    .fillna('unknown') # <-- Se for nulo, vira 'unknown'
)

cols_dimensao = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in cols_dimensao:
    products[col] = products.groupby('product_category_name_english')[col].transform(
        lambda x: x.fillna(x.median())
    )

print("Quantidade de nulos após o tratamento CORRIGIDO:")
print(products[cols_dimensao].isnull().sum())

Quantidade de nulos após o tratamento CORRIGIDO:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [9]:
# Carregando as tabelas que faltavam para a criação da Tabela Master

customers = pd.read_csv(RAW_PATH + 'olist_customers_dataset.csv')
order_items = pd.read_csv(RAW_PATH + 'olist_order_items_dataset.csv')
sellers = pd.read_csv(RAW_PATH + 'olist_sellers_dataset.csv')
reviews = pd.read_csv(RAW_PATH + 'olist_order_reviews_dataset.csv')
payments = pd.read_csv(RAW_PATH + 'olist_order_payments_dataset.csv')

print("Todas as tabelas de suporte foram carregadas com sucesso na memória!")

Todas as tabelas de suporte foram carregadas com sucesso na memória!


In [10]:
# Base: orders (pedido é a unidade de análise)
df_master = orders.copy()

# JOIN 1: customers — left join para manter todos os pedidos
df_master = df_master.merge(customers, on='customer_id', how='left')

# JOIN 2: order_items — Um pedido pode ter vários itens
# Agregar os itens por pedido primeiro para evitar duplicação de linhas
items_agg = order_items.groupby('order_id').agg(
    qtd_itens=('order_item_id', 'count'),
    valor_total_produtos=('price', 'sum'),
    valor_total_frete=('freight_value', 'sum'),
    seller_id_principal=('seller_id', 'first')  # vendedor do primeiro item
).reset_index()

df_master = df_master.merge(items_agg, on='order_id', how='left')

# JOIN 3: sellers (trazendo informações do vendedor principal do pedido)
df_master = df_master.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    left_on='seller_id_principal',
    right_on='seller_id',
    how='left'
)

# JOIN 4: reviews — Pode ter múltiplas reviews por pedido (raro, mas existe)
# Pegar apenas a review mais recente por pedido
reviews_dedup = reviews.sort_values('review_creation_date').drop_duplicates(
    subset='order_id', keep='last'
)
df_master = df_master.merge(
    reviews_dedup[['order_id', 'review_score', 'review_comment_message']],
    on='order_id',
    how='left'
)

# JOIN 5: payments — agregar por pedido
payments_agg = payments.groupby('order_id').agg(
    tipo_pagamento_principal=('payment_type', 'first'),
    parcelas=('payment_installments', 'max'),
    valor_pago_total=('payment_value', 'sum')
).reset_index()

df_master = df_master.merge(payments_agg, on='order_id', how='left')

# VERIFICAÇÃO: o número de linhas não deve ter mudado (continua 1 linha por pedido)
print(f"Linhas na tabela master: {len(df_master):,}")
print(f"Linhas originais em orders: {len(orders):,}")
assert len(df_master) == len(orders), "ERRO: join gerou duplicatas! Revise a lógica."

print("\nShape final da master table:", df_master.shape)

Linhas na tabela master: 99,441
Linhas originais em orders: 99,441

Shape final da master table: (99441, 34)


In [11]:
# ── Salvando a Master Table na pasta de dados processados em PARQUET ─────────

PROCESSED_PATH = '../data/processed/'

# Exportando para Parquet (usando o pyarrow)
df_master.to_parquet(PROCESSED_PATH + 'master_table.parquet', engine='pyarrow', index=False)

# Salvando também a tabela de produtos
products.to_parquet(PROCESSED_PATH + 'products_clean.parquet', engine='pyarrow', index=False)

print("SUCESSO! Dados limpos e salvos em formato PARQUET na pasta data/processed/.")

SUCESSO! Dados limpos e salvos em formato PARQUET na pasta data/processed/.
